# 2. URLs, Endpoints, and Methods

A web API request has a few important pieces:

- **URL**: the full address we send the request to.
- **Base URL**: the main API address.
- **Endpoint**: the specific path for the data or action we want.
- **Query parameters**: extra options after a `?` in the URL.
- **Method**: the action, such as `GET` or `POST`.


In [1]:
import requests
from pprint import pprint
from urllib.parse import urlparse, parse_qs

TIMEOUT = 10

## Reading a URL

This URL asks Open-Meteo's geocoding API to search for Los Angeles.


In [2]:
url = "https://geocoding-api.open-meteo.com/v1/search?name=Los%20Angeles&count=1"
parts = urlparse(url)

print("scheme:", parts.scheme)
print("domain:", parts.netloc)
print("endpoint/path:", parts.path)
print("query string:", parts.query)
print("query parameters:", parse_qs(parts.query))

scheme: https
domain: geocoding-api.open-meteo.com
endpoint/path: /v1/search
query string: name=Los%20Angeles&count=1
query parameters: {'name': ['Los Angeles'], 'count': ['1']}


## Build URLs With `params`

Instead of typing the whole query string ourselves, we can give `requests.get()` a dictionary of parameters.


In [3]:
geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
params = {
    "name": "Los Angeles",
    "count": 1,
}

response = requests.get(geocoding_url, params=params, timeout=TIMEOUT)

print(response.status_code)
print(response.url)

200
https://geocoding-api.open-meteo.com/v1/search?name=Los+Angeles&count=1


In [4]:
location_data = response.json()
pprint(location_data)

{'generationtime_ms': 1.3641119,
 'results': [{'admin1': 'California',
              'admin1_id': 5332921,
              'admin2': 'Los Angeles',
              'admin2_id': 5368381,
              'country': 'United States',
              'country_code': 'US',
              'country_id': 6252001,
              'elevation': 89.0,
              'feature_code': 'PPLA2',
              'id': 5368361,
              'latitude': 34.05223,
              'longitude': -118.24368,
              'name': 'Los Angeles',
              'population': 3820914,
              'postcodes': ['90001',
                            '90002',
                            '90003',
                            '90004',
                            '90005',
                            '90006',
                            '90007',
                            '90008',
                            '90009',
                            '90010',
                            '90011',
                            '90012',
         

The API returned a list of matching locations. We only asked for one result, so we can use the first item.


In [5]:
first_result = location_data["results"][0]

city = first_result["name"]
latitude = first_result["latitude"]
longitude = first_result["longitude"]

print(city)
print(latitude, longitude)

Los Angeles
34.05223 -118.24368


## Same API Family, Different Endpoint

Now we will use the latitude and longitude with Open-Meteo's forecast endpoint.


In [6]:
forecast_url = "https://api.open-meteo.com/v1/forecast"
forecast_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,wind_speed_10m",
}

forecast_response = requests.get(forecast_url, params=forecast_params, timeout=TIMEOUT)

print(forecast_response.status_code)
print(forecast_response.url)

200
https://api.open-meteo.com/v1/forecast?latitude=34.05223&longitude=-118.24368&current=temperature_2m%2Cwind_speed_10m


In [7]:
forecast = forecast_response.json()
current = forecast["current"]
units = forecast["current_units"]

print(f"Temperature: {current['temperature_2m']} {units['temperature_2m']}")
print(f"Wind speed: {current['wind_speed_10m']} {units['wind_speed_10m']}")

Temperature: 16.8 °C
Wind speed: 6.7 km/h


## Common HTTP Methods

The method tells the API what kind of action we want.


In [8]:
methods = {
    "GET": "Read or request data",
    "POST": "Send new data to the server",
    "PUT": "Replace existing data",
    "PATCH": "Update part of existing data",
    "DELETE": "Delete data",
}

for method, meaning in methods.items():
    print(f"{method}: {meaning}")

GET: Read or request data
POST: Send new data to the server
PUT: Replace existing data
PATCH: Update part of existing data
DELETE: Delete data


In this course we will practice `GET` and `POST`, because they are the methods beginners see most often.
